# Phase 2 — Measure

## 05 — Store Dimension Creation

### Objective

The objective of this notebook is to create a governed **Store Dimension (`dim_store`)** from the store-level inventory data contained in `fact_stock`.

The original Stock-on-Hand dataset was transformed from wide format into a long, store-level structure during the earlier Phase 2 transformation. This notebook extracts the unique store entities from that governed stock data and assigns each store a stable surrogate key (`store_id`).

The generated `store_id` will then be mapped back into `fact_stock`, allowing the inventory fact table to connect to `dim_store` through a standard dimension-to-fact relationship.

---

## Input

- `fact_stock.csv`

The existing `fact_stock` table contains store-level stock observations and the previously established `product_id`.

---

## Outputs

This notebook will produce:

1. **`dim_store.csv`**
   - One row per unique store
   - Contains a unique `store_id`
   - Used as the Store Dimension in the analytical data model

2. **Updated `fact_stock.csv`**
   - Retains the existing stock records
   - Adds `store_id` as the foreign key linking each stock record to `dim_store`

---

## Target Data Model

The resulting relationship will be:

`dim_store[store_id]` **1 → many** `fact_stock[store_id]`

This complements the existing product relationship:

`dim_product[product_id]` **1 → many** `fact_stock[product_id]`

Together, these relationships support store-level inventory analysis within the Power BI star schema.

---

## Dimension Grain

**`dim_store`: One row per unique retail store**

Each `store_id` must uniquely identify one store.

---

## Validation Criteria

The transformation will be considered successful when:

- Every unique store in `fact_stock` exists exactly once in `dim_store`
- `store_id` is unique and contains no null values in `dim_store`
- Every row in `fact_stock` successfully maps to a valid `store_id`
- No stock records are lost during the mapping process
- Existing `product_id` values and stock quantities remain unchanged
- The number of unique stores in `fact_stock` matches the number of rows in `dim_store`

---

## Phase 2 Role

This notebook completes the governed dimensional structure required for downstream analytics.

Final governed tables:

- `dim_product`
- `dim_store`
- `fact_sales`
- `fact_stock`
- `fact_soa`

These tables form the governed analytical layer consumed by Phase 4 — Power BI Decision Dashboard.

In [1]:
## Load existing processed fact_stock
import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parents[1]

# Processed data folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Existing fact_stock
fact_stock_path = PROCESSED_DIR / "fact_stock.csv"

fact_stock = pd.read_csv(fact_stock_path)

print("Loaded:", fact_stock_path)
print("Shape:", fact_stock.shape)
print("Columns:", fact_stock.columns.tolist())

fact_stock.head()

Loaded: d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\fact_stock.csv
Shape: (2681, 8)
Columns: ['Product_ID', 'Model', 'Store', 'Category', 'Description', 'Quantity', 'Stock_Status', 'Outstanding_Order_Qty']


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0


In [2]:
dim_store = (
    fact_stock[["Store"]]
    .drop_duplicates()
    .sort_values("Store")
    .reset_index(drop=True)
)

dim_store.insert(
    0,
    "store_id",
    range(1, len(dim_store) + 1)
)

dim_store

,store_id,Store
0,1,Belfast
1,2,Blanch
2,3,Cavan
3,4,Dundrum
4,5,Gorey
5,6,Navan
6,7,Sandyford


In [3]:
## Add store_id to existing fact_stock
rows_before = len(fact_stock)

fact_stock_updated = fact_stock.merge(
    dim_store,
    on="Store",
    how="left"
)

rows_after = len(fact_stock_updated)

print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Missing store_id:", fact_stock_updated["store_id"].isna().sum())

fact_stock_updated.head()

Rows before: 2681
Rows after : 2681
Missing store_id: 0


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty,store_id
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0,1
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,2
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,3
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0,4
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0,5


In [4]:
print("=== DIM STORE VALIDATION ===")

print("Rows:", len(dim_store))
print("Unique store_id:", dim_store["store_id"].nunique())
print("Unique stores:", dim_store["Store"].nunique())
print("Duplicate store_id:", dim_store["store_id"].duplicated().sum())
print("Duplicate stores:", dim_store["Store"].duplicated().sum())


print("\n=== FACT STOCK VALIDATION ===")

print("Original rows:", len(fact_stock))
print("Updated rows:", len(fact_stock_updated))

print(
    "Missing store_id:",
    fact_stock_updated["store_id"].isna().sum()
)

print(
    "Unique stores:",
    fact_stock_updated["Store"].nunique()
)

print(
    "Unique store_id:",
    fact_stock_updated["store_id"].nunique()
)

=== DIM STORE VALIDATION ===
Rows: 7
Unique store_id: 7
Unique stores: 7
Duplicate store_id: 0
Duplicate stores: 0

=== FACT STOCK VALIDATION ===
Original rows: 2681
Updated rows: 2681
Missing store_id: 0
Unique stores: 7
Unique store_id: 7


In [5]:
fact_stock_updated[
    ["store_id", "Store"]
].drop_duplicates().sort_values("store_id")

,store_id,Store
0,1,Belfast
1,2,Blanch
2,3,Cavan
3,4,Dundrum
4,5,Gorey
5,6,Navan
6,7,Sandyford


In [ ]:
# Save new dimension

""" dim_store.to_csv(
    PROCESSED_DIR / "dim_store.csv",
    index=False
)

# Replace existing fact_stock with updated version
fact_stock_updated.to_csv(
    PROCESSED_DIR / "fact_stock.csv",
    index=False
)

print("Saved successfully:")
print(PROCESSED_DIR / "dim_store.csv")
print(PROCESSED_DIR / "fact_stock.csv") """

Saved successfully:
d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\dim_store.csv
d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\fact_stock.csv
